In [ ]:
from pathlib import Path
import os
import numpy as np
from cpymad.madx import Madx
from scipy.constants import physical_constants

# Constants
e0 = physical_constants['electron mass energy equivalent in MeV'][0] * 1e6  # eV
energy = 20e9  # eV

# Paths
workdir = Path("/home/martinez/ML4CollEffects/notebooks/ext_HEB/optics/v24_1")
lattice_file = workdir / "heb_ring_z.madx"
tfs_file = workdir / "heb_ring_z.tfs"

# Start MAD-X
mad = Madx()
os.chdir(workdir)

# Load lattice and sequence
mad.call(str(lattice_file))

mad.input(f"""
beam, particle=electron, energy={energy/1e9};
use, sequence=fccee_heb_ring_v24_1;
""")

# Run TWISS and save to TFS
#mad.twiss(file=str(tfs_file), centre=True)

#print(f"TFS written to: {tfs_file}")

# Access TWISS in memory (reference)
twiss_mem = mad.table.twiss
s_mem = np.array(twiss_mem.s)
betx_mem = np.array(twiss_mem.betx)
bety_mem = np.array(twiss_mem.bety)

print(f"Number of points: {len(s_mem)}")
print(f"s range: {s_mem[0]} -> {s_mem[-1]}")
print(f"betx mean: {betx_mem.mean():.3f}")
print(f"bety mean: {bety_mem.mean():.3f}")

In [ ]:
# Reset MAD-X (clean state)
mad = Madx()

# Reload lattice and sequence BEFORE reading table
mad.call(str(lattice_file))

mad.input(f"""
beam, particle=electron, energy={energy/1e9};
use, sequence=fccee_heb_ring_v24_1;
""")

# Read the table
mad.readtable(file=str(tfs_file), table="twiss_from_file")

twiss_file = mad.table.twiss_from_file
s_file = np.array(twiss_file.s)
betx_file = np.array(twiss_file.betx)

# Consistency check
diff = np.max(np.abs(betx_mem - betx_file))
print(f"Max difference between memory and file: {diff:.3e}")

if diff < 1e-9:
    print("TFS file is consistent with MAD-X result")
else:
    print("Something is inconsistent!")

In [ ]:
dx_mem = np.array(twiss_mem['dx'])
dy_mem = np.array(twiss_mem['dy'])

In [ ]:
import matplotlib.pyplot as plt 

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# 1. Beta Functions
ax1.plot(s_mem, betx_mem, label='Beta X', color='blue', linewidth=1.5)
ax1.plot(s_mem, bety_mem, label='Beta Y', color='red', linewidth=1.5)
ax1.set_ylabel('Beta Function (m)', fontsize=12)
ax1.set_title('Beta Functions vs. Path Length (s)', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.set_ylim(bottom=0) # Beta functions are positive

# 2. Dispersion (if available)
if dx_mem is not None:
    ax2.plot(s_mem, dx_mem, label='Dispersion X (Dx)', color='green', linewidth=1.5)
    ax2.plot(s_mem, dy_mem, label='Dispersion Y (Dy)', color='orange', linewidth=1.5)
    ax2.set_ylabel('Dispersion (m)', fontsize=12)
    ax2.set_xlabel('Path Length (m)', fontsize=12)
    ax2.set_title('Dispersion Functions', fontsize=14)
    ax2.legend(fontsize=10)
    ax2.grid(True, linestyle='--', alpha=0.6)
else:
    ax2.text(0.5, 0.5, 'Dispersion not found.\nTo calculate it:\n1. Add "twiss, disp;" to your input\n2. Or check if "dx" exists in the table.', 
             ha='center', va='center', transform=ax2.transAxes, fontsize=12, color='gray')
    ax2.set_ylabel('Dispersion (m)')
    ax2.set_xlabel('Path Length (m)')

plt.tight_layout()
plt.show()

In [ ]:
# Requires: cpymad, xtrack, numpy
import warnings
from pathlib import Path
from typing import Optional, Tuple, Union

import numpy as np
from cpymad.madx import Madx
import xtrack as xt


def build_line_from_madx(
    madx_file: str,
    seq_name: str,
    p0c_ev: Optional[float] = 1e9,
    mass0_ev: Optional[float] = None,
    verbose: bool = False,
) -> xt.Line:
    """
    Build an xtrack.Line from a MAD-X file sequence using cpymad, mapping common elements.

    Parameters
    ----------
    madx_file : str
        Path to the .mad / .madx file.
    seq_name : str
        Sequence name in the MAD-X file to import (e.g. "fccee_heb_ring_v24_1").
    p0c_ev : float, optional
        Reference momentum in eV (default 1e9).
    mass0_ev : float, optional
        Particle rest energy in eV (default: xt.PROTON_MASS_EV if available).
    verbose : bool
        Print mapping information and warnings.

    Returns
    -------
    xtrack.Line
        Constructed xtrack Line with tracker built.

    Notes
    -----
    - This is a pragmatic mapper for common elements (DRIFT, QUADRUPOLE, SBEND/RBEND, MARKER, MULTIPOLE).
    - Unrecognized elements are replaced by a Drift(length) placeholder with a warning.
    - Validate Twiss/optics after import if fidelity is critical; mapping of k1 conventions may need adjustment.
    """
    madx_path = Path(madx_file)
    if not madx_path.exists():
        raise FileNotFoundError(f"MAD-X file not found: {madx_file!r}")

    mad = Madx()
    mad.call(str(madx_path))

    if seq_name not in mad.sequence:
        raise RuntimeError(f"Sequence {seq_name!r} not found in MAD-X after loading {madx_path}")

    seq = mad.sequence[seq_name]

    try:
        order = list(seq.element_names)
    except Exception:
        order = list(seq.elements.keys())

    xt_elements = []
    element_names = []

    for ename in order:
        elem = seq.elements[ename]
        
        etype_raw = getattr(elem, "_type", None) or getattr(elem, "type", None)
        etype = str(etype_raw).lower() if etype_raw is not None else ""
        length = float(getattr(elem, "l", 0.0) or 0.0)
        k0 = getattr(elem, "k0", None)
        k1 = getattr(elem, "k1", None)
        k2 = getattr(elem, "k2", None)
        angle = getattr(elem, "angle", None)
        h = getattr(elem, "h", None)

        if "drift" in etype or etype.strip() == "":
            xel = xt.Drift(length=length)

        elif "quad" in etype or "quadrupole" in etype:
            k1f = float(k1) if k1 is not None else 0.0
            
            xel = xt.Quadrupole(k1=k1f, length=length)

        elif "rbend" in etype or "sbend" in etype or "bend" in etype or "dipole" in etype:
            a = float(angle) if angle is not None else 0.0
            xel = xt.Sbend(angle=a, length=length)

        elif "marker" in etype:
            xel = xt.Marker()

        elif "multipole" in etype or "sext" in etype or "sextupole" in etype or ("k2" in etype):
            knl = {}
            if k0 is not None:
                knl[0] = float(k0)
            if k1 is not None:
                knl[1] = float(k1)
            if k2 is not None:
                knl[2] = float(k2)
            if hasattr(xt, "Multipole"):
                try:
                    xel = xt.Multipole(knl=knl)
                except Exception:
                    if verbose:
                        warnings.warn(f"Multipole mapping for '{ename}' failed; using Drift({length}) fallback")
                    xel = xt.Drift(length=length)
            else:
                xel = xt.Drift(length=length)

        elif "rf" in etype or "cavity" in etype:
            if verbose:
                warnings.warn(f"RF/cavity element '{ename}' encountered — inserting Drift(length={length}) placeholder")
            xel = xt.Drift(length=length)

        else:
            if verbose:
                warnings.warn(f"Element '{ename}' type '{etype}' not mapped; using Drift(length={length}) fallback")
            xel = xt.Drift(length=length)

        xt_elements.append(xel)
        element_names.append(str(ename))

    # Build xtrack line
    xline = xt.Line(elements=xt_elements, element_names=element_names)

    # set reference particle
    if mass0_ev is None:
        mass0_ev = getattr(xt, "PROTON_MASS_EV", None)
        if mass0_ev is None:
            # fallback rough proton mass in eV
            mass0_ev = 938272088.0
    xline.particle_ref = xt.Particles(mass0=float(mass0_ev), p0c=float(p0c_ev))

    xline.build_tracker()

    if verbose:
        print(f"[build_line_from_madx] Imported {len(xt_elements)} elements from {madx_file} seq={seq_name}")

    return xline


# ###################### IMPEDANCE ##################################
def build_wake_kernel(zeta_grid: np.ndarray, wake_cfg) -> np.ndarray:
    """Fallback wake builder (keeps your earlier semantics)."""
    z = zeta_grid - zeta_grid.mean()
    kind = getattr(wake_cfg, "kind", "gaussian")
    if kind == "gaussian":
        W = np.exp(-0.5 * (z / wake_cfg.sigma) ** 2)
    elif kind == "exponential":
        W = np.exp(-wake_cfg.decay * np.maximum(z, 0.0))
    elif kind == "resonator":
        W = np.sin(2 * np.pi * wake_cfg.freq * z) * np.exp(-np.abs(z) / wake_cfg.sigma)
    else:
        raise ValueError(f"Unknown wake kind: {kind}")
    W = float(getattr(wake_cfg, "strength", 1.0)) * W
    
    W = W / (np.abs(W).sum() + 1e-12)
    return W


def load_impedance(
    impedance: Optional[Union[str, Tuple[np.ndarray, np.ndarray]]],
    density_cfg,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Resolve an impedance/wake specification into (zeta_grid, W).
    impedance may be:
      - None: returns (None, None)
      - tuple (zeta_grid, W)
      - path to .npz (keys: 'zeta_grid' and 'W' or 'wake') or .npy (W only)
      - 1D numpy array W (then zeta_grid from density_cfg is used)
    """
    if impedance is None:
        return None, None

    if isinstance(impedance, tuple) and len(impedance) == 2:
        zg, W = impedance
        return np.asarray(zg), np.asarray(W)

    if isinstance(impedance, np.ndarray):
        nz = density_cfg.nz
        zg = np.linspace(density_cfg.zeta_min, density_cfg.zeta_max, nz)
        return zg, impedance

    if isinstance(impedance, str) or isinstance(impedance, Path):
        p = Path(impedance)
        if not p.exists():
            raise FileNotFoundError(f"Impedance file not found: {impedance}")
        if p.suffix == ".npz":
            d = np.load(str(p))
            if "zeta_grid" in d and ("W" in d or "wake" in d):
                zg = d["zeta_grid"]
                W = d.get("W", d.get("wake"))
                return np.asarray(zg), np.asarray(W)
            if "z" in d and "W" in d:
                return np.asarray(d["z"]), np.asarray(d["W"])
            raise RuntimeError(f".npz does not contain recognized keys (zeta_grid,W) in {p}")
        elif p.suffix == ".npy":
            W = np.load(str(p))
            nz = density_cfg.nz
            zg = np.linspace(density_cfg.zeta_min, density_cfg.zeta_max, nz)
            return zg, np.asarray(W)
        else:
            raise RuntimeError(f"Unsupported impedance file extension: {p.suffix}")

    raise RuntimeError("Unsupported impedance spec. Provide (zeta_grid,W), np.array(W), or filepath to .npz/.npy")


def track_with_collective_effects(
    line: xt.Line,
    z0: np.ndarray,
    mu: np.ndarray,
    density_cfg,
    wake_cfg: Optional[object] = None,
    impedance: Optional[Union[str, Tuple[np.ndarray, np.ndarray], np.ndarray]] = None,
    eps: float = 1e-12,
) -> np.ndarray:
    """
    Track a particle cloud through 'line' then apply a longitudinal collective kick
    derived from either wake_cfg (fallback) or a provided impedance/wake model.

    Parameters
    ----------
    line : xtrack.Line
        Tracker line to use for external transport.
    z0 : ndarray, shape [Np,6]
        Input cloud.
    mu : ndarray-like
        Parameter vector [Q_bunch, pipe_radius, impedance_scale] (used for scaling).
    density_cfg: object
        An object with attributes nz, zeta_min, zeta_max (same as your DensityGridConfig).
    wake_cfg : optional
        If impedance is None, wake_cfg is used to build a kernel (build_wake_kernel).
    impedance : optional
        If provided, used in preference to wake_cfg. Accepts (zeta_grid,W) tuple, path to .npz/.npy, or W array.
    eps : float
        small numerical constant.

    Returns
    -------
    z1 : ndarray shape [Np,6]
        Tracked cloud after external transport and delta kicks applied.
    """
    Q, a, Z_scale = mu

    p = line.build_particles(
        x=z0[:, 0],
        y=z0[:, 1],
        zeta=z0[:, 2],
        px=z0[:, 3],
        py=z0[:, 4],
        delta=z0[:, 5],
    )
    line.track(p)
    z1 = np.column_stack(
        [
            np.asarray(p.x),
            np.asarray(p.y),
            np.asarray(p.zeta),
            np.asarray(p.px),
            np.asarray(p.py),
            np.asarray(p.delta),
        ]
    ).astype(np.float64)

    if impedance is None and wake_cfg is None:
        return z1

    nz = density_cfg.nz
    zeta_grid, lambda_z = None, None
    zeta_grid, lambda_z = line_density_from_cloud(z1, density_cfg)

    zg_imp, W_imp = load_impedance(impedance, density_cfg)
    if zg_imp is None:
        if wake_cfg is None:
            raise RuntimeError("No wake_cfg provided and impedance not given; cannot compute wake kernel.")
        W = build_wake_kernel(zeta_grid, wake_cfg)
        zg = zeta_grid
    else:
        zg = np.asarray(zg_imp)
        W = np.asarray(W_imp)
        if zg.shape != zeta_grid.shape or not np.allclose(zg, zeta_grid):
            W = np.interp(zeta_grid, zg, W)
            zg = zeta_grid

    W = W / (np.abs(W).sum() + eps)

    wake_conv = np.fft.ifft(np.fft.fft(lambda_z) * np.fft.fft(W)).real

    delta_kick = np.interp(z1[:, 2], zeta_grid, wake_conv)

    delta_kick *= (Q * Z_scale) / (a ** 2 + eps)

    z1[:, 5] += delta_kick

    return z1

In [ ]:
# scripts/madx_import.py
from pathlib import Path
from typing import List, Optional, Tuple, Union
import warnings

def get_madx_sequences(madx_file: str) -> List[str]:
    """
    Return the list of sequence names defined in madx_file using cpymad.
    Raises FileNotFoundError if file missing or RuntimeError if MAD-X fails to load.
    """
    from cpymad.madx import Madx
    p = Path(madx_file)
    if not p.exists():
        raise FileNotFoundError(f"MAD-X file not found: {madx_file!r}")
    mad = Madx()
    mad.call(str(p))
    # mad.sequence is a dict-like mapping name->Sequence
    seq_names = list(mad.sequence.keys())
    return seq_names


def build_line_from_madx_auto(
    madx_file: str,
    seq_name: Optional[str] = None,
    seq_index: int = 0,
    p0c_ev: Optional[float] = 1e9,
    mass0_ev: Optional[float] = None,
    verbose: bool = False,
):
    """
    Auto-import MAD-X sequence into an xtrack.Line.
    - If seq_name is provided, use it.
    - Otherwise, discover sequences in the file and pick seq_index (default 0).
    - Prints available sequences when verbose=True.

    Returns: xtrack.Line (tracker built)
    """
    from cpymad.madx import Madx
    import xtrack as xt
    from pathlib import Path
    import numpy as np

    p = Path(madx_file)
    if not p.exists():
        raise FileNotFoundError(f"MAD-X file not found: {madx_file!r}")

    # load MAD-X and get sequences
    mad = Madx()
    mad.call(str(p))
    seq_names = list(mad.sequence.keys())

    if verbose:
        print(f"[get_madx_sequences] found {len(seq_names)} sequences: {seq_names}")

    if not seq_names:
        # Try to parse file text as fallback
        text = p.read_text()
        import re
        found = re.findall(r"sequence, *name *= *([A-Za-z0-9_]+)", text, flags=re.IGNORECASE)
        if found:
            seq_names = found
            if verbose:
                print(f"[fallback] parsed sequence names from file text: {seq_names}")

    if seq_name is None:
        if not seq_names:
            raise RuntimeError(f"No sequences found in {madx_file!r}. You must provide seq_name.")
        if seq_index < 0 or seq_index >= len(seq_names):
            raise IndexError(f"seq_index {seq_index} out of range (0..{len(seq_names)-1})")
        chosen = seq_names[seq_index]
        if verbose:
            print(f"[build_line_from_madx_auto] no seq_name given; selecting sequence index {seq_index}: {chosen}")
    else:
        if seq_name not in seq_names:
            # accept case-insensitive match
            matches = [s for s in seq_names if s.lower() == seq_name.lower()]
            if matches:
                chosen = matches[0]
                if verbose:
                    print(f"[build_line_from_madx_auto] using case-insensitive match: {chosen}")
            else:
                # If provided name not present, show available and raise
                raise ValueError(f"Sequence {seq_name!r} not found in file. Available: {seq_names}")
        else:
            chosen = seq_name

    # Now convert chosen sequence to xtrack.Line using a pragmatic mapper.
    # (This uses the mapping logic from earlier; keep it short here by calling the mapper)
    xline = _build_xtrack_line_from_madx_sequence(mad, chosen, p0c_ev=p0c_ev, mass0_ev=mass0_ev, verbose=verbose)
    return xline


def _build_xtrack_line_from_madx_sequence(mad: "Madx", seq_name: str, p0c_ev: Optional[float], mass0_ev: Optional[float], verbose: bool):
    """
    Internal: map the loaded cpymad.Madx instance 'mad' and sequence name to an xtrack.Line.
    (This is the same pragmatic mapper as discussed; supports common element types.)
    """
    import xtrack as xt
    import numpy as np
    import warnings

    if seq_name not in mad.sequence:
        raise RuntimeError(f"Sequence {seq_name!r} not found in MAD-X instance")

    seq = mad.sequence[seq_name]
    # element order
    try:
        order = list(seq.element_names)
    except Exception:
        order = list(seq.elements.keys())

    xt_elements = []
    element_names = []
    for ename in order:
        elem = seq.elements[ename]
        etype_raw = getattr(elem, "_type", None) or getattr(elem, "type", None)
        etype = str(etype_raw).lower() if etype_raw is not None else ""
        length = float(getattr(elem, "l", 0.0) or 0.0)
        k0 = getattr(elem, "k0", None)
        k1 = getattr(elem, "k1", None)
        k2 = getattr(elem, "k2", None)
        angle = getattr(elem, "angle", None)
        # mapping
        if "drift" in etype or etype.strip() == "":
            xel = xt.Drift(length=length)
        elif "quad" in etype:
            k1f = float(k1) if k1 is not None else 0.0
            xel = xt.Quadrupole(k1=k1f, length=length)
        elif "rbend" in etype or "sbend" in etype or "bend" in etype or "dipole" in etype:
            a = float(angle) if angle is not None else 0.0
            xel = xt.Sbend(angle=a, length=length)
        elif "marker" in etype:
            xel = xt.Marker()
        elif "multipole" in etype or "sext" in etype or "sextupole" in etype:
            knl = {}
            if k0 is not None:
                knl[0] = float(k0)
            if k1 is not None:
                knl[1] = float(k1)
            if k2 is not None:
                knl[2] = float(k2)
            if hasattr(xt, "Multipole"):
                try:
                    xel = xt.Multipole(knl=knl)
                except Exception:
                    if verbose:
                        warnings.warn(f"Multipole mapping failed for {ename}; using Drift({length})")
                    xel = xt.Drift(length=length)
            else:
                xel = xt.Drift(length=length)
        elif "rf" in etype or "cavity" in etype:
            if verbose:
                warnings.warn(f"RF/cavity '{ename}' found — inserting Drift({length}) placeholder")
            xel = xt.Drift(length=length)
        else:
            if verbose:
                warnings.warn(f"Element '{ename}' type '{etype}' not mapped; using Drift(length={length})")
            xel = xt.Drift(length=length)

        xt_elements.append(xel)
        element_names.append(str(ename))

    xline = xt.Line(elements=xt_elements, element_names=element_names)
    # set reference particle
    if mass0_ev is None:
        mass0_ev = getattr(xt, "PROTON_MASS_EV", 938272088.0)
    xline.particle_ref = xt.Particles(mass0=float(mass0_ev), p0c=float(p0c_ev))
    xline.build_tracker()
    if verbose:
        print(f"[mapper] Built xtrack.Line for sequence {seq_name} with {len(xt_elements)} elements")
    return xline

In [ ]:

print(get_madx_sequences("/home/martinez/ML4CollEffects/notebooks/ext_HEB/optics/v24_1/heb_ring_z.madx"))

In [ ]:
from pathlib import Path
import os
import numpy as np
from cpymad.madx import Madx
from scipy.constants import physical_constants
from pathlib import Path
import warnings
from cpymad.madx import Madx
import xtrack as xt
import numpy as np

from contextlib import contextmanager
from pathlib import Path
import os

from __future__ import annotations

import os
from pathlib import Path
from contextlib import contextmanager
from typing import Optional, Sequence

from cpymad.madx import Madx


@contextmanager
def pushd(path: Path):
    old = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)


def _is_missing_file_error(exc: Exception, filename: str) -> bool:
    msg = str(exc).lower()
    return ("cannot open input file" in msg) and (filename.lower() in msg)


def run_madx_driver_with_bootstrap(
    workdir: str | Path,
    driver_madx: str,
    *,
    bootstrap_madx: Optional[str] = None,
    bootstrap_candidates: Sequence[str] = (
        "generate.madx",
        "make.madx",
        "build.madx",
        "export.madx",
        "run.madx",
    ),
    verbose: bool = True,
) -> Madx:
    """
    Run a MAD-X driver script in workdir, retrying once if sidecar files are missing.
    If missing, optionally run bootstrap_madx to generate sidecars, then retry.

    Parameters
    ----------
    workdir:
        Directory containing your MAD-X files.
    driver_madx:
        The main file to call (e.g. "heb_ring_z.madx").
    bootstrap_madx:
        A MAD-X script that generates sidecar files (heb_ring_z.str/.seq/.ele).
        If None, tries bootstrap_candidates in workdir.
    bootstrap_candidates:
        Filenames to try if bootstrap_madx is not given.
    """
    workdir = Path(workdir).expanduser().resolve()
    driver_path = (workdir / driver_madx).resolve()
    if not driver_path.exists():
        raise FileNotFoundError(f"Driver MAD-X file not found: {driver_path}")

    def call_file(mad: Madx, file_path: Path):
        file_path = Path(file_path).resolve()
        with pushd(workdir):
            if verbose:
                print(f"[MAD-X] python_cwd={Path.cwd()}")
                print(f"[MAD-X] calling_abs={file_path}")
                print(f"[MAD-X] exists={file_path.exists()} readable={os.access(file_path, os.R_OK)}")
            mad.call(str(file_path))   # <-- ABSOLUTE PATH

    mad = Madx()
    try:
        call_file(mad, driver_path)
        return mad
    except Exception as e:
        # Detect the specific missing sidecar file (you can add more here)
        missing_sidecar = None
        for f in (f"{driver_path.stem}.str", f"{driver_path.stem}.seq", f"{driver_path.stem}.ele"):
            if _is_missing_file_error(e, f):
                missing_sidecar = f
                break

        if missing_sidecar is None:
            # Not the missing-sidecar case -> re-raise
            raise RuntimeError(f"MAD-X failed running {driver_path.name}: {e}") from e

        if verbose:
            print(f"[MAD-X] detected missing sidecar: {missing_sidecar}")

        # 2) Determine bootstrap script to run
        if bootstrap_madx is not None:
            boot_path = (workdir / bootstrap_madx).resolve()
            if not boot_path.exists():
                raise FileNotFoundError(f"bootstrap_madx not found: {boot_path}")
        else:
            boot_path = None
            for cand in bootstrap_candidates:
                p = (workdir / cand)
                if p.exists():
                    boot_path = p.resolve()
                    break

        if boot_path is None:
            raise FileNotFoundError(
                f"MAD-X driver requires sidecar file {missing_sidecar!r} but it is missing.\n"
                f"Workdir: {workdir}\n"
                f"Driver: {driver_path}\n\n"
                f"Provide a bootstrap MAD-X script that generates the sidecars "
                f"(pass bootstrap_madx='...'), or generate/copy {missing_sidecar} into the workdir."
            )

        # 3) Run bootstrap then retry driver
        if verbose:
            print(f"[MAD-X] running bootstrap: {boot_path.name}")
        mad2 = Madx()
        call_file(mad2, boot_path)

        # re-run driver with a fresh MAD-X instance (clean state)
        mad3 = Madx()
        call_file(mad3, driver_path)
        return mad3



def build_line_from_madx(
    workdir : str,
    madx_file: str,
    seq_name: str,
    p0c_ev: float = 1e9,
    mass0_ev: float | None = None,
    verbose: bool = False,
) -> "xtrack.Line":
    """
    Robust importer: load MAD-X via cpymad, find sequence (seq_name),
    obtain element order (handles multiple cpymad variants), map common
    MAD-X elements to xtrack elements, build tracker and return xtrack.Line.

    Parameters
    ----------
    madx_file : str
        Path to MAD-X file.
    seq_name : str
        Name of sequence to import.
    p0c_ev : float
        Reference momentum (eV).
    mass0_ev : float | None
        Particle rest energy in eV (if None, fallback to xt.PROTON_MASS_EV).
    verbose : bool
        Print mapping info/warnings.
    """
    

    # Start MAD-X
    mad = run_madx_driver_with_bootstrap(
        workdir=workdir,
        driver_madx=madx_file,
        bootstrap_madx=None,   # or set to the actual generator file if you have it
        verbose=True,
    )
    if seq_name not in mad.sequence:
        # try case-insensitive match
        seqs = list(mad.sequence.keys())
        lower_map = {s.lower(): s for s in seqs}
        if seq_name.lower() in lower_map:
            seq_name = lower_map[seq_name.lower()]
            if verbose:
                print(f"[build_line_from_madx] using case-insensitive match -> '{seq_name}'")
        else:
            raise RuntimeError(f"Sequence {seq_name!r} not found. Available: {seqs}")

    seq = mad.sequence[seq_name]

    # Obtain element order robustly
    names = []
    # 1) preferred: element_names() method
    try:
        en = getattr(seq, "element_names", None)
        if callable(en):
            names = list(en())
            if verbose:
                print(f"[import] used seq.element_names() -> {len(names)} elements")
    except Exception:
        names = []

    # 2) try iterating seq.elements (ElementList is iterable in some cpymad versions)
    if not names:
        try:
            for elem in seq.elements:
                # try to extract a name attribute
                nm = getattr(elem, "name", None)
                if nm is None:
                    # fallback: cpymad sometimes exposes element as a tuple-like repr
                    nm = str(elem).split()[0]
                names.append(str(nm))
            if verbose:
                print(f"[import] obtained names by iterating seq.elements -> {len(names)} elements")
        except Exception:
            names = []

    # 3) fallback: inspect internal attrs of seq.elements
    if not names:
        elems_obj = seq.elements
        # try common internal storage attribute names
        possible_attrs = ["_list", "_elems", "_dict", "_elements"]
        found = False
        for a in possible_attrs:
            if hasattr(elems_obj, a):
                container = getattr(elems_obj, a)
                try:
                    # container may be list-like or dict-like
                    if isinstance(container, dict):
                        names = list(container.keys())
                    else:
                        # assume list-like of element objects
                        for e in container:
                            nm = getattr(e, "name", None) or str(e).split()[0]
                            names.append(str(nm))
                    found = True
                    if verbose:
                        print(f"[import] obtained names from seq.elements.{a} -> {len(names)} elements")
                    break
                except Exception:
                    continue
        if not found:
            # last resort: try attributes() introspection
            try:
                # seq.elements may support indexing by integer
                i = 0
                while True:
                    e = seq.elements[i]  # may raise
                    nm = getattr(e, "name", None) or str(e).split()[0]
                    names.append(str(nm))
                    i += 1
            except Exception:
                pass

    if not names:
        # give up with helpful diagnostic
        raise RuntimeError(
            "Could not determine element order from MAD-X sequence. "
            "Try inspecting mad.sequence keys and seq.elements in an interactive session."
        )

    # Build xtrack elements mapping
    xt_elements = []
    element_names = []
    for ename in names:
        # access element object robustly
        try:
            elem = seq.elements[ename]
        except Exception:
            # seq.elements may not support indexing by name; find by scanning
            found_elem = None
            try:
                for e in seq.elements:
                    nm = getattr(e, "name", None) or str(e).split()[0]
                    if str(nm) == str(ename):
                        found_elem = e
                        break
            except Exception:
                found_elem = None
            if found_elem is None:
                # if still not found, warn and insert a zero-length drift placeholder
                warnings.warn(f"[build_line_from_madx] element {ename!r} not accessible; inserting Drift(0.0)")
                xt_elements.append(xt.Drift(length=0.0))
                element_names.append(str(ename))
                continue
            elem = found_elem

        etype_raw = getattr(elem, "_type", None) or getattr(elem, "type", None)
        etype = str(etype_raw).lower() if etype_raw is not None else ""
        length = float(getattr(elem, "l", 0.0) or 0.0)
        k0 = getattr(elem, "k0", None)
        k1 = getattr(elem, "k1", None)
        k2 = getattr(elem, "k2", None)
        angle = getattr(elem, "angle", None)

        # mapping decisions
        try:
            if "drift" in etype or etype.strip() == "":
                xel = xt.Drift(length=length)

            elif "quad" in etype or "quadrupole" in etype:
                k1f = float(k1) if k1 is not None else 0.0
                xel = xt.Quadrupole(k1=k1f, length=length)

            elif "rbend" in etype or "sbend" in etype or "bend" in etype or "dipole" in etype:
                a = float(angle) if angle is not None else 0.0
                xel = xt.Sbend(angle=a, length=length)

            elif "marker" in etype:
                xel = xt.Marker()

            elif "multipole" in etype or "sext" in etype or "sextupole" in etype:
                knl = {}
                if k0 is not None:
                    knl[0] = float(k0)
                if k1 is not None:
                    knl[1] = float(k1)
                if k2 is not None:
                    knl[2] = float(k2)
                if hasattr(xt, "Multipole"):
                    try:
                        xel = xt.Multipole(knl=knl)
                    except Exception:
                        if verbose:
                            warnings.warn(f"Multipole mapping for '{ename}' failed; using Drift({length})")
                        xel = xt.Drift(length=length)
                else:
                    xel = xt.Drift(length=length)

            elif "rf" in etype or "cavity" in etype:
                if verbose:
                    warnings.warn(f"RF/cavity element '{ename}' encountered — inserting Drift(length={length}) placeholder")
                xel = xt.Drift(length=length)

            else:
                if verbose:
                    warnings.warn(f"Element '{ename}' type '{etype}' not explicitly mapped; using Drift(length={length}) fallback")
                xel = xt.Drift(length=length)

        except Exception as ex:
            warnings.warn(f"Error mapping element '{ename}': {ex}; using Drift(length={length}) fallback")
            xel = xt.Drift(length=length)

        xt_elements.append(xel)
        element_names.append(str(ename))

    # Create xtrack.Line
    xline = xt.Line(elements=xt_elements, element_names=element_names)

    # particle ref
    if mass0_ev is None:
        mass0_ev = getattr(xt, "PROTON_MASS_EV", 938272088.0)
    xline.particle_ref = xt.Particles(mass0=float(mass0_ev), p0c=float(p0c_ev))

    # build tracker
    xline.build_tracker()

    if verbose:
        print(f"[build_line_from_madx] built xtrack.Line with {len(xt_elements)} elements from seq '{seq_name}'")

    return xline



In [8]:
xline = build_line_from_madx(workdir="/home/martinez/ML4CollEffects/notebooks/ext_HEB/optics/v24_1",
                             madx_file= "heb_ring_z.madx",
                             seq_name="fcc_heb",
                             p0c_ev=1e9,
                             verbose=True)
print("elements:", len(xline.elements))
print("first 10 names:", xline.element_names[:10])


  ++++++++++++++++++++++++++++++++++++++++++++
  +     MAD-X 5.09.03  (64 bit, Linux)       +
  + Support: mad@cern.ch, http://cern.ch/mad +
  + Release   date: 2024.04.25               +
  + Execution date: 2026.05.05 13:48:30      +
  ++++++++++++++++++++++++++++++++++++++++++++
[MAD-X] cwd=/home/martinez/ML4CollEffects/notebooks/ext_HEB/optics/v24_1 call=heb_ring_z.madx


+=+=+= fatal: cannot open input file: heb_ring_z.madx


RuntimeError: MAD-X failed running heb_ring_z.madx: MAD-X has stopped working!

In [ ]:
from pathlib import Path
from typing import Dict, Iterable, List, Literal, Optional, Tuple
from dataclasses import asdict, dataclass


PHASE_SPACE_DIM = 6
PHASE_SPACE_LABELS = ("x", "y", "zeta", "px", "py", "delta")

@dataclass
class BeamFamilyConfig:
    family: Literal[
        "gaussian",
        "correlated_gaussian",
        "core_halo",
        "mixture",
        "mismatched"
    ]
    x_sigma: float = 1e-3
    y_sigma: float = 1e-3
    zeta_sigma: float = 1e-3
    px_sigma: float = 1e-3
    py_sigma: float = 1e-3
    delta_sigma: float = 1e-4
    halo_scale: float = 4.0
    halo_fraction: float = 0.15
    mismatch_scale: float = 2.0
    corr_strength: float = 0.45
    mixture_shift_scale: float = 1.2


@dataclass
class LatticeConfig:
    p0c_ev: float = 1e9
    mass0_ev: float = xt.PROTON_MASS_EV
    l_mq: float = 0.4
    line_length: float = 4.0
    qf1_at: float = 1.0
    qd1_at: float = 2.0
    qf2_at: float = 3.0

@dataclass
class WakeConfig:
    kind: Literal["gaussian", "exponential", "resonator"] = "gaussian"
    strength: float = 1.0
    sigma: float = 1e-3
    decay: float = 1e3
    freq: float = 1e10

@dataclass
class ParameterRanges:
    bunch_charge_range: Tuple[float, float] = (0.1e-9, 3e-9)   # Coulombs
    pipe_radius_range: Tuple[float, float] = (5e-3, 30e-3)     # meters
    impedance_scale_range: Tuple[float, float] = (0.1, 10.0)   # dimensionless


@dataclass
class DensityGridConfig:
    nz: int = 256
    zeta_min: float = -5e-3
    zeta_max: float = 5e-3
    normalize_density: bool = True


@dataclass
class DatasetConfig:
    n_samples: int = 512
    particles_per_sample: int = 4096
    seed: int = 42
    output_dir: str = "./data/neural"
    save_cloud_dataset: bool = True
    save_density_dataset: bool = True
    save_moments: bool = True
    train_fraction: float = 0.8
    val_fraction: float = 0.1

def build_wake_kernel(zeta_grid, wake_cfg):
    z = zeta_grid - zeta_grid.mean()

    if wake_cfg.kind == "gaussian":
        W = np.exp(-0.5 * (z / wake_cfg.sigma)**2)

    elif wake_cfg.kind == "exponential":
        W = np.exp(-wake_cfg.decay * np.maximum(z, 0.0))

    elif wake_cfg.kind == "resonator":
        W = np.sin(2*np.pi*wake_cfg.freq*z) * np.exp(-np.abs(z)/wake_cfg.sigma)

    else:
        raise ValueError

    return wake_cfg.strength * W / (np.abs(W).sum() + 1e-12)

def apply_wake(lambda_z, W):
    n = len(lambda_z)
    return np.convolve(lambda_z, W, mode="same")


def build_line(lattice_cfg: LatticeConfig) -> Tuple[xt.Line, xt.Environment]:
    """Build a simple Xsuite line with three quadrupoles."""
    part_ref = xt.Particles(mass0=lattice_cfg.mass0_ev, p0c=lattice_cfg.p0c_ev)

    env = xt.Environment()
    env.vars.default_to_zero = True
    env["l_mq"] = lattice_cfg.l_mq

    env.new("mq", xt.Quadrupole, length="l_mq")
    env.new("qf1", "mq", k1="kf1")
    env.new("qd1", "mq", k1="kd1")
    env.new("qf2", "mq", k1="kf2")
    env.new("start_cell", xt.Marker)
    env.new("end_cell", xt.Marker)

    line = env.new_line(
        length=lattice_cfg.line_length,
        components=[
            env.place("start_cell", at=0.0),
            env.place("qf1", at=lattice_cfg.qf1_at),
            env.place("qd1", at=lattice_cfg.qd1_at),
            env.place("qf2", at=lattice_cfg.qf2_at),
            env.place("end_cell", at=lattice_cfg.line_length),
        ],
    )
    line.particle_ref = part_ref
    line.build_tracker()
    return line, env


def _diag_sigmas(cfg: BeamFamilyConfig) -> np.ndarray:
    return np.array(
        [
            cfg.x_sigma,
            cfg.y_sigma,
            cfg.zeta_sigma,
            cfg.px_sigma,
            cfg.py_sigma,
            cfg.delta_sigma,
        ],
        dtype=np.float64,
    )


def sample_initial_conditions(
    n_particles: int,
    rng: np.random.Generator,
    beam_cfg: BeamFamilyConfig,
) -> np.ndarray:
    """Sample a particle cloud in R^6.

    Output order is [x, y, zeta, px, py, delta].
    """
    sigmas = _diag_sigmas(beam_cfg)

    if beam_cfg.family == "gaussian":
        z = rng.normal(0.0, sigmas, size=(n_particles, PHASE_SPACE_DIM))
        return z.astype(np.float64)

    if beam_cfg.family == "correlated_gaussian":
        cov = np.diag(sigmas ** 2)
        # modest correlations in each plane
        corr = beam_cfg.corr_strength
        cov[0, 3] = cov[3, 0] = corr * sigmas[0] * sigmas[3]
        cov[1, 4] = cov[4, 1] = corr * sigmas[1] * sigmas[4]
        cov[2, 5] = cov[5, 2] = corr * sigmas[2] * sigmas[5]
        z = rng.multivariate_normal(np.zeros(PHASE_SPACE_DIM), cov, size=n_particles)
        return z.astype(np.float64)

    if beam_cfg.family == "core_halo":
        n_halo = int(round(beam_cfg.halo_fraction * n_particles))
        n_core = n_particles - n_halo
        core = rng.normal(0.0, sigmas, size=(n_core, PHASE_SPACE_DIM))
        halo = rng.normal(0.0, beam_cfg.halo_scale * sigmas, size=(n_halo, PHASE_SPACE_DIM))
        z = np.vstack([core, halo])
        rng.shuffle(z, axis=0)
        return z.astype(np.float64)

    if beam_cfg.family == "mixture":
        shift = beam_cfg.mixture_shift_scale * sigmas
        n_a = n_particles // 2
        n_b = n_particles - n_a
        a = rng.normal(-shift, sigmas, size=(n_a, PHASE_SPACE_DIM))
        b = rng.normal(+shift, sigmas, size=(n_b, PHASE_SPACE_DIM))
        z = np.vstack([a, b])
        rng.shuffle(z, axis=0)
        return z.astype(np.float64)

    if beam_cfg.family == "mismatched":
        scaled = np.array(sigmas)
        scaled[[0, 1, 3, 4]] *= beam_cfg.mismatch_scale
        z = rng.normal(0.0, scaled, size=(n_particles, PHASE_SPACE_DIM))
        return z.astype(np.float64)

    raise ValueError(f"Unsupported beam family: {beam_cfg.family}")


def particles_to_6d(particles: xt.Particles) -> np.ndarray:
    return np.column_stack(
        [
            np.asarray(particles.x),
            np.asarray(particles.y),
            np.asarray(particles.zeta),
            np.asarray(particles.px),
            np.asarray(particles.py),
            np.asarray(particles.delta),
        ]
    ).astype(np.float64)


def sample_parameters(
    rng: np.random.Generator,
    param_ranges: ParameterRanges,
) -> np.ndarray:
    return np.array(
        [
            rng.uniform(*param_ranges.bunch_charge_range),
            rng.uniform(*param_ranges.pipe_radius_range),
            rng.uniform(*param_ranges.impedance_scale_range),
        ],
        dtype=np.float64,
    )

"""def set_lattice_parameters(env: xt.Environment, mu: np.ndarray) -> None:
    env["kf1"] = float(mu[0])
    env["kd1"] = float(mu[1])
    env["kf2"] = float(mu[2])
"""

def track_cloud(line: xt.Line, z0: np.ndarray) -> np.ndarray:
    p = line.build_particles(
        x=z0[:, 0],
        y=z0[:, 1],
        zeta=z0[:, 2],
        px=z0[:, 3],
        py=z0[:, 4],
        delta=z0[:, 5],
    )
    line.track(p)
    return particles_to_6d(p)


def track_with_collective_effects(
    line: xt.Line,
    z0: np.ndarray,
    mu: np.ndarray,
    density_cfg,
    wake_cfg: Optional[object] = None,
    impedance: Optional[Union[str, Tuple[np.ndarray, np.ndarray], np.ndarray]] = None,
    eps: float = 1e-12,
) -> np.ndarray:
    """
    Track a particle cloud through 'line' then apply a longitudinal collective kick
    derived from either wake_cfg (fallback) or a provided impedance/wake model.

    Parameters
    ----------
    line : xtrack.Line
        Tracker line to use for external transport.
    z0 : ndarray, shape [Np,6]
        Input cloud.
    mu : ndarray-like
        Parameter vector [Q_bunch, pipe_radius, impedance_scale] (used for scaling).
    density_cfg: object
        An object with attributes nz, zeta_min, zeta_max (same as your DensityGridConfig).
    wake_cfg : optional
        If impedance is None, wake_cfg is used to build a kernel (build_wake_kernel).
    impedance : optional
        If provided, used in preference to wake_cfg. Accepts (zeta_grid,W) tuple, path to .npz/.npy, or W array.
    eps : float
        small numerical constant.

    Returns
    -------
    z1 : ndarray shape [Np,6]
        Tracked cloud after external transport and delta kicks applied.
    """
    Q, a, Z_scale = mu

    # --- track externally
    p = line.build_particles(
        x=z0[:, 0],
        y=z0[:, 1],
        zeta=z0[:, 2],
        px=z0[:, 3],
        py=z0[:, 4],
        delta=z0[:, 5],
    )
    line.track(p)
    z1 = np.column_stack(
        [
            np.asarray(p.x),
            np.asarray(p.y),
            np.asarray(p.zeta),
            np.asarray(p.px),
            np.asarray(p.py),
            np.asarray(p.delta),
        ]
    ).astype(np.float64)

    # If no wake/impedance specified, return purely tracked cloud
    if impedance is None and wake_cfg is None:
        return z1

    # --- compute zeta grid & input density
    nz = density_cfg.nz
    # If the density grid is inside the dataset .npz, prefer that; else build from density_cfg
    zeta_grid, lambda_z = None, None
    zeta_grid, lambda_z = line_density_from_cloud(z1, density_cfg)

    # --- get wake kernel W (either loaded or built)
    zg_imp, W_imp = load_impedance(impedance, density_cfg)
    if zg_imp is None:
        # fallback: build from wake_cfg using same grid
        if wake_cfg is None:
            raise RuntimeError("No wake_cfg provided and impedance not given; cannot compute wake kernel.")
        W = build_wake_kernel(zeta_grid, wake_cfg)
        zg = zeta_grid
    else:
        # If the provided wake grid differs from lambda grid, resample/align via interpolation
        zg = np.asarray(zg_imp)
        W = np.asarray(W_imp)
        if zg.shape != zeta_grid.shape or not np.allclose(zg, zeta_grid):
            # Interpolate provided W onto the lambda grid
            W = np.interp(zeta_grid, zg, W)
            zg = zeta_grid

    # normalize W similarly to previous code
    W = W / (np.abs(W).sum() + eps)

    # --- convolution (FFT-based circular conv with same length)
    wake_conv = np.fft.ifft(np.fft.fft(lambda_z) * np.fft.fft(W)).real

    # --- interpolate kick back to particle z positions
    delta_kick = np.interp(z1[:, 2], zeta_grid, wake_conv)

    # --- scaling (keeping your previous scaling convention)
    delta_kick *= (Q * Z_scale) / (a ** 2 + eps)

    # apply kick to delta
    z1[:, 5] += delta_kick

    return z1



def cloud_moments(z: np.ndarray) -> Dict[str, np.ndarray]:
    centroid = z.mean(axis=0)
    cov = np.cov(z, rowvar=False)
    return {"centroid": centroid, "cov": cov}


def line_density_from_cloud(
    z: np.ndarray,
    density_cfg: DensityGridConfig,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return (bin_centers, normalized_histogram) for zeta marginal."""
    hist, edges = np.histogram(
        z[:, 2],
        bins=density_cfg.nz,
        range=(density_cfg.zeta_min, density_cfg.zeta_max),
        density=False,
    )
    centers = 0.5 * (edges[:-1] + edges[1:])
    hist = hist.astype(np.float64)
    if density_cfg.normalize_density:
        dz = (density_cfg.zeta_max - density_cfg.zeta_min) / density_cfg.nz
        norm = hist.sum() * dz
        if norm > 0:
            hist = hist / norm
    return centers, hist


def build_datasets(
    line: xt.Line,
    env: xt.Environment,
    dataset_cfg: DatasetConfig,
    density_cfg: DensityGridConfig,
    param_ranges: ParameterRanges,
    beam_families: Iterable[BeamFamilyConfig],
    use_collective: bool,
    wake_cfg: Optional[WakeConfig]
) -> Dict[str, np.ndarray]:
    rng = np.random.default_rng(dataset_cfg.seed)
    families = list(beam_families)
    if not families:
        raise ValueError("At least one beam family must be provided.")

    cloud_in, cloud_out, mu_all = [], [], []
    lambda_in, lambda_out, zeta_grid = [], [], None
    moments_in_centroid, moments_out_centroid = [], []
    moments_in_cov, moments_out_cov = [], []
    family_id = []

    for _ in range(dataset_cfg.n_samples):
        beam_cfg = families[rng.integers(0, len(families))]
        mu = sample_parameters(rng, param_ranges)
        
        z0 = sample_initial_conditions(
            dataset_cfg.particles_per_sample,
            rng,
            beam_cfg
        )

        if use_collective:
            z1 = track_with_collective_effects(
                line,
                z0,
                mu,
                density_cfg,
                wake_cfg,
            )
        else:
            z1 = track_cloud(line, z0)

        
       

        if dataset_cfg.save_cloud_dataset:
            cloud_in.append(z0.astype(np.float32))
            cloud_out.append(z1.astype(np.float32))
            mu_all.append(mu.astype(np.float32))

        if dataset_cfg.save_density_dataset:
            grid, lam0 = line_density_from_cloud(z0, density_cfg)
            _, lam1 = line_density_from_cloud(z1, density_cfg)
            zeta_grid = grid.astype(np.float32)
            lambda_in.append(lam0.astype(np.float32))
            lambda_out.append(lam1.astype(np.float32))

        if dataset_cfg.save_moments:
            m0 = cloud_moments(z0)
            m1 = cloud_moments(z1)
            moments_in_centroid.append(m0["centroid"].astype(np.float32))
            moments_out_centroid.append(m1["centroid"].astype(np.float32))
            moments_in_cov.append(m0["cov"].astype(np.float32))
            moments_out_cov.append(m1["cov"].astype(np.float32))

        family_id.append(beam_cfg.family)

    out: Dict[str, np.ndarray] = {}
    if cloud_in:
        out["X_cloud"] = np.stack(cloud_in, axis=0)
        out["Y_cloud"] = np.stack(cloud_out, axis=0)
        out["MU"] = np.stack(mu_all, axis=0)
    if lambda_in:
        out["zeta_grid"] = zeta_grid
        out["X_lambda"] = np.stack(lambda_in, axis=0)
        out["Y_lambda"] = np.stack(lambda_out, axis=0)
        if "MU" not in out:
            out["MU"] = np.stack(mu_all, axis=0)
    if moments_in_centroid:
        out["centroid_in"] = np.stack(moments_in_centroid, axis=0)
        out["centroid_out"] = np.stack(moments_out_centroid, axis=0)
        out["cov_in"] = np.stack(moments_in_cov, axis=0)
        out["cov_out"] = np.stack(moments_out_cov, axis=0)

    out["family_id"] = np.array(family_id)
    return out


def split_indices(
    n: int,
    train_fraction: float,
    val_fraction: float,
    rng: np.random.Generator,
) -> Dict[str, np.ndarray]:
    if not (0 < train_fraction < 1):
        raise ValueError("train_fraction must be in (0, 1).")
    if not (0 <= val_fraction < 1):
        raise ValueError("val_fraction must be in [0, 1).")
    if train_fraction + val_fraction >= 1:
        raise ValueError("train_fraction + val_fraction must be < 1.")

    perm = rng.permutation(n)
    n_train = int(round(train_fraction * n))
    n_val = int(round(val_fraction * n))
    train_idx = perm[:n_train]
    val_idx = perm[n_train:n_train + n_val]
    test_idx = perm[n_train + n_val:]
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def save_dataset_bundle(
    data: Dict[str, np.ndarray],
    dataset_cfg: DatasetConfig,
    density_cfg: DensityGridConfig,
    beam_families: Iterable[BeamFamilyConfig],
    param_ranges: ParameterRanges,
) -> None:
    out_dir = Path(dataset_cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    n_samples = None
    for key, value in data.items():
        if isinstance(value, np.ndarray) and value.ndim > 0 and value.shape[0] > 1 and key != "zeta_grid":
            n_samples = value.shape[0]
            break
    if n_samples is None:
        raise RuntimeError("Could not infer dataset size.")

    rng = np.random.default_rng(dataset_cfg.seed + 17)
    split = split_indices(n_samples, dataset_cfg.train_fraction, dataset_cfg.val_fraction, rng)

    time_of_generation = np.datetime64("now").astype(str)
    filename = f"neural_xsuite_dataset_{time_of_generation}.npz"
    np.savez(out_dir / filename, **data, **split)

    metadata = {
        "phase_space_labels": PHASE_SPACE_LABELS,
        "dataset_config": asdict(dataset_cfg),
        "density_grid_config": asdict(density_cfg),
        "parameter_ranges": asdict(param_ranges),
        "beam_families": [asdict(b) for b in beam_families],
        "keys": sorted(list(data.keys()) + list(split.keys())),
    }
    (out_dir / "xsuite_neural_dataset_metadata.json").write_text(
        json.dumps(metadata, indent=2)
    )


def default_beam_families() -> List[BeamFamilyConfig]:
    return [
        BeamFamilyConfig(family="gaussian"),
        BeamFamilyConfig(family="correlated_gaussian"),
        BeamFamilyConfig(family="core_halo"),
        BeamFamilyConfig(family="mixture"),
        BeamFamilyConfig(family="mismatched"),
    ]



madx_path = "/home/martinez/ML4CollEffects/notebooks/ext_HEB/optics/v24_1/heb_ring_z.madx"
seq_name = "fcc_heb"

line = xline

density_cfg = DensityGridConfig(nz=256, zeta_min=-5e-3, zeta_max=5e-3)
rng = np.random.default_rng(42)

beam_families = default_beam_families()
families = list(beam_families)

beam_cfg = families[rng.integers(0, len(families))]
dataset_cfg = DatasetConfig(
        n_samples=1,
        particles_per_sample=50000,
        seed=42,
        output_dir="/home/martinez/ML4CollEffects/notebooks/ext_HEB/optics/output",
        save_cloud_dataset= True,
        save_density_dataset= True,
        train_fraction=0.8,
        val_fraction=0.2,
    )



param_ranges = ParameterRanges()
mu = sample_parameters(rng, param_ranges)    

z0 = sample_initial_conditions(
            dataset_cfg.particles_per_sample,
            rng,
            beam_cfg
        )

my_wake_cfg = WakeConfig(
        kind="exponential",     # "gaussian" | "exponential" | "resonator"
        strength=1.0,
        sigma=1e-3,
    )

# Option A: use wake_cfg fallback (existing)
z1 = track_with_collective_effects(line, z0, mu, density_cfg, wake_cfg=my_wake_cfg)

# Option B: use precomputed impedance file (npz contains 'zeta_grid' and 'W' arrays)
#z1 = track_with_collective_effects(line, z0, mu, density_cfg, impedance="/path/to/wake_model.npz")

In [ ]:
#!/usr/bin/env python3
# scripts/plot_transport_distributions.py

from pathlib import Path
import os
import math
import json
from typing import Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt

try:
    from data_generator_neural import line_density_from_cloud
except Exception:
    line_density_from_cloud = None


def ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)


def dz_from_grid(z: np.ndarray) -> float:
    return float((z[-1] - z[0]) / max(1, z.size - 1))


def compute_1d_moments(arr: np.ndarray, grid: Optional[np.ndarray] = None, eps: float = 1e-12) -> Dict[str, float]:
    """
    If grid is provided, interpret arr as samples on that grid (density values).
    If grid is None, interpret arr as sample points (1D particle coordinate array).
    Returns stats dictionary.
    """
    out = {}
    if grid is None:
        a = np.asarray(arr).astype(np.float64)
        out["n"] = a.size
        if a.size == 0:
            out.update({"mass": 0.0, "mean": 0.0, "sigma": 0.0, "skew": 0.0, "kurt": 0.0, "entropy": 0.0})
            return out
        out["mass"] = float(a.size)
        out["mean"] = float(a.mean())
        var = float(a.var())
        out["sigma"] = float(math.sqrt(max(var, 0.0)))
        if out["sigma"] > 0:
            out["skew"] = float(((a - out["mean"])**3).mean() / (out["sigma"]**3 + eps))
            out["kurt"] = float(((a - out["mean"])**4).mean() / (out["sigma"]**4 + eps))
        else:
            out["skew"], out["kurt"] = 0.0, 0.0
        out["entropy"] = 0.0
    else:
        lam = np.asarray(arr).astype(np.float64)
        z = np.asarray(grid).astype(np.float64)
        dz = dz_from_grid(z)
        mass = float(lam.sum() * dz)
        lamn = lam / (mass + eps)
        mean = float((z * lamn).sum() * dz)
        var = float(((z - mean)**2 * lamn).sum() * dz)
        sigma = float(math.sqrt(max(var, 0.0)))
        if sigma > 0:
            m3 = float(((z - mean)**3 * lamn).sum() * dz)
            m4 = float(((z - mean)**4 * lamn).sum() * dz)
            skew = float(m3 / (sigma**3 + eps))
            kurt = float(m4 / (sigma**4 + eps))
        else:
            skew, kurt = 0.0, 0.0
        p = np.clip(lamn, eps, None)
        entropy = float(-(p * np.log(p)).sum() * dz)
        out.update({
            "mass": mass,
            "mean": mean,
            "sigma": sigma,
            "skew": skew,
            "kurt": kurt,
            "entropy": entropy,
        })
    return out


def plot_pair_marginals(z_before: np.ndarray, z_after: np.ndarray, out_path: str,
                        bins: int = 200, cmap: str = "viridis", figsize=(10, 12)):
    """
    z_before / z_after: arrays shape (N,6)
    Creates 3 rows x 2 cols: before/after hist2d for (x,px), (y,py), (zeta,delta)
    """
    ensure_dir(os.path.dirname(out_path) or ".")
    pairs = [(0, 3, "x", "px"), (1, 4, "y", "py"), (2, 5, "zeta", "delta")]
    fig, axes = plt.subplots(3, 2, figsize=figsize, constrained_layout=True)
    for r, (i, j, xl, yl) in enumerate(pairs):
        # compute ranges that cover both before and after
        combined = np.vstack([z_before[:, [i, j]], z_after[:, [i, j]]])
        xmin, xmax = combined[:, 0].min(), combined[:, 0].max()
        ymin, ymax = combined[:, 1].min(), combined[:, 1].max()
        xpad = 1e-12 + 0.05 * (xmax - xmin)
        ypad = 1e-12 + 0.05 * (ymax - ymin)
        xr = (xmin - xpad, xmax + xpad)
        yr = (ymin - ypad, ymax + ypad)

        im0 = axes[r, 0].hist2d(z_before[:, i], z_before[:, j], bins=bins, range=[xr, yr], density=True, cmap=cmap)[3]
        axes[r, 0].set_title(f"Before: {xl} vs {yl}")
        fig.colorbar(im0, ax=axes[r, 0], shrink=0.8)

        im1 = axes[r, 1].hist2d(z_after[:, i], z_after[:, j], bins=bins, range=[xr, yr], density=True, cmap=cmap)[3]
        axes[r, 1].set_title(f"After: {xl} vs {yl}")
        fig.colorbar(im1, ax=axes[r, 1], shrink=0.8)

    fig.suptitle("2D marginals before / after transport")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_longitudinal_density(z_before: np.ndarray, z_after: np.ndarray, out_path: str,
                              nz: int = 256, zmin: float = -5e-3, zmax: float = 5e-3,
                              figsize=(10, 4)):
    """
    Plot normalized longitudinal density (lambda) before & after.
    """
    ensure_dir(os.path.dirname(out_path) or ".")
    z0 = z_before[:, 2]
    z1 = z_after[:, 2]
    centers = np.linspace(zmin, zmax, nz)
    hist0, edges = np.histogram(z0, bins=nz, range=(zmin, zmax), density=False)
    hist1, _ = np.histogram(z1, bins=nz, range=(zmin, zmax), density=False)
    dz = (zmax - zmin) / nz
    norm0 = hist0.sum() * dz
    norm1 = hist1.sum() * dz
    dens0 = hist0 / (norm0 + 1e-12)
    dens1 = hist1 / (norm1 + 1e-12)

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.plot(centers, dens0, label="before")
    ax.plot(centers, dens1, label="after")
    ax.set_xlabel("zeta")
    ax.set_ylabel("lambda(zeta) (normalized)")
    ax.set_title("Longitudinal density")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    return centers, dens0, dens1


def plot_density_images(z_before: np.ndarray, z_after: np.ndarray, out_path: str,
                        grid_n: int = 64, sigma_steps: float = 2.0, figsize=(10, 8)):
    """
    Quick 2D KDE images for (x,px), (y,py), (zeta,delta) using Gaussian smoothing on a grid.
    Useful for qualitative checks when FieldBuilder is not available.
    """
    from scipy.ndimage import gaussian_filter  # requires scipy
    ensure_dir(os.path.dirname(out_path) or ".")

    pairs = [((0, 3), "x,px"), ((1, 4), "y,py"), ((2, 5), "zeta,delta")]
    fig, axes = plt.subplots(3, 2, figsize=figsize, constrained_layout=True)

    def kde_image(samples, ix, iy):
        x = samples[:, ix]
        y = samples[:, iy]
        # grid bounds
        xmin, xmax = x.min(), x.max()
        ymin, ymax = y.min(), y.max()
        if xmin == xmax:
            xmax = xmin + 1e-6
        if ymin == ymax:
            ymax = ymin + 1e-6
        xi = np.linspace(xmin, xmax, grid_n)
        yi = np.linspace(ymin, ymax, grid_n)
        H, _, _ = np.histogram2d(x, y, bins=[xi, yi])
        # gaussian smooth
        sigma = sigma_steps
        Hs = gaussian_filter(H, sigma=sigma)
        return xi, yi, Hs.T  # transpose so imshow displays correctly

    for r, (idx_pair, title) in enumerate(pairs):
        ix, iy = idx_pair
        xi, yi, Hb = kde_image(z_before, ix, iy)
        xi, yi, Ha = kde_image(z_after, ix, iy)

        im0 = axes[r, 0].imshow(Hb, origin="lower", aspect="auto", extent=[xi[0], xi[-1], yi[0], yi[-1]], cmap="viridis")
        axes[r, 0].set_title(f"Before {title}")
        fig.colorbar(im0, ax=axes[r, 0], fraction=0.046)

        im1 = axes[r, 1].imshow(Ha, origin="lower", aspect="auto", extent=[xi[0], xi[-1], yi[0], yi[-1]], cmap="viridis")
        axes[r, 1].set_title(f"After {title}")
        fig.colorbar(im1, ax=axes[r, 1], fraction=0.046)

    fig.suptitle("KDE images before / after")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# --- convenience: compute and save summary JSON
def summarize_and_save(z_before: np.ndarray, z_after: np.ndarray, out_dir: str,
                       density_cfg=None):
    ensure_dir(out_dir)
    summary = {}
    # particle-based moments for each projection
    projections = [("x", 0), ("y", 1), ("zeta", 2), ("px", 3), ("py", 4), ("delta", 5)]
    for name, idx in projections:
        summary[f"{name}_before"] = compute_1d_moments(z_before[:, idx])
        summary[f"{name}_after"] = compute_1d_moments(z_after[:, idx])

    # longitudinal density moments using density_cfg if available
    if density_cfg is not None and line_density_from_cloud is not None:
        zgrid_b, lam_b = line_density_from_cloud(z_before, density_cfg)
        zgrid_a, lam_a = line_density_from_cloud(z_after, density_cfg)
        summary["lambda_before"] = compute_1d_moments(lam_b, zgrid_b)
        summary["lambda_after"] = compute_1d_moments(lam_a, zgrid_a)
    else:
        # fallback: use histogram-based density
        centers, dens0, dens1 = plot_longitudinal_density(z_before, z_after, out_path=os.path.join(out_dir, "lambda_plot.png"))
        summary["lambda_before"] = compute_1d_moments(dens0, centers)
        summary["lambda_after"] = compute_1d_moments(dens1, centers)

    json_path = os.path.join(out_dir, "transport_summary.json")
    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2)
    return summary


# --- Example top-level runner
def run_plots_for_sample(z_before: np.ndarray, z_after: np.ndarray, out_dir: str = "./output", sample_id: int = 0,
                         density_cfg=None):
    ensure_dir(out_dir)
    plot_pair_marginals(z_before, z_after, out_path=os.path.join(out_dir, f"pairs_sample_{sample_id}.png"))
    plot_longitudinal_density(z_before, z_after, out_path=os.path.join(out_dir, f"lambda_sample_{sample_id}.png"))
    try:
        plot_density_images(z_before, z_after, out_path=os.path.join(out_dir, f"kde_images_sample_{sample_id}.png"))
    except Exception:
        # scipy may not be installed; ignore
        pass
    summary = summarize_and_save(z_before, z_after, out_dir, density_cfg=density_cfg)
    print(f"[OK] plots and summary written to {out_dir}")
    return summary



In [ ]:
run_plots_for_sample(z0, z1, out_dir="./output", sample_id=0, density_cfg=density_cfg)